# XMCA-Net — Complete Research Pipeline & Benchmark Execution

**Cross-Modal Consistency Attribution Network for Detecting and Diagnosing Hallucinations in RAG-VLMs**

- **Hardware Requirement**: Kaggle Dual NVIDIA T4 (2 × 16GB VRAM)
- **Datasets**: POPE (1,000), FaithScore (174), A-OKVQA (1,000), InfoSeek (1,000) — Total 3,174 frozen records
- **Backbones**: LLaVA-1.5-7B, Qwen2-VL-7B, InternVL2-2B
- **Diagnostics**: Perception Consistency (S_P), Retrieval Consistency (S_R), Utilization Consistency (S_U)
- **Combiner**: Calibrated XMCA Combiner with Missingness Masks & Anti-Leakage Partitioning
- **Attribution**: Ground-Truth Supervised Cause Attribution Head (Perception vs Retrieval vs Utilization)
- **Strict Rule**: Zero synthetic data, zero proxy scores, hardware execution strictly on Dual T4 GPUs.

## 1. Hardware Environment & Dual NVIDIA T4 Verification

In [ ]:
import os
import sys
import gc
import time
import torch

# Prevent CUDA memory fragmentation across long-horizon generation runs
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"GPU Count:       {gpu_count}")

if gpu_count < 2:
    print("WARNING: dual NVIDIA T4 GPUs are required for the full benchmark.")
else:
    for i in range(gpu_count):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name} | Total VRAM: {props.total_memory / (1024**3):.2f} GiB")

!nvidia-smi


## 2. Environment Setup & Repository Synchronization

In [ ]:
# Clone or sync repository
import os
import sys
from pathlib import Path

repo_dir = Path("/kaggle/working/XMCA_NET")
if not repo_dir.exists():
    !git clone -b feature/xmca-full-pipeline https://github.com/Venkat262005/XMCA_NET.git /kaggle/working/XMCA_NET
else:
    !git -C /kaggle/working/XMCA_NET pull origin feature/xmca-full-pipeline

%cd /kaggle/working/XMCA_NET
os.environ["PYTHONPATH"] = "/kaggle/working/XMCA_NET:" + os.environ.get("PYTHONPATH", "")
if "/kaggle/working/XMCA_NET" not in sys.path:
    sys.path.insert(0, "/kaggle/working/XMCA_NET")

# Install dependencies
!pip install -q "transformers<=4.49.0" "sentencepiece<0.2.2"
!pip install -q -r requirements.txt
!pip install -q pyarrow
print("Environment initialized and synced with feature/xmca-full-pipeline.")


## 2.1 Benchmark Tier & Experiment Configuration
Configure the progressive evaluation tier (100, 300, 600, or 1000) and optional scale experiments.

In [ ]:
# ========================================================
# RESEARCH BENCHMARK EXECUTION CONFIGURATION
# ========================================================
import os
TARGET_TIER = 1000  # Full benchmark tier: 1,000 POPE, 1,000 InfoSeek, 1,000 A-OKVQA, 174 FaithScore (3,174 total)
os.environ["TARGET_TIER"] = str(TARGET_TIER)
RUN_SCALE_EXPERIMENT = False  # Set to True to evaluate InternVL2-8B scale test

print(f"Target Benchmark Tier:          {TARGET_TIER} samples/dataset")
print(f"InternVL2-8B Scale Experiment:  {RUN_SCALE_EXPERIMENT}")


## 3. Authentic Cohort & Manifest Verification

In [ ]:
import pandas as pd

cohort_df = pd.read_parquet("artifacts/manifests/cohort_manifest.parquet")
print("Cohort distribution across datasets:")
print(cohort_df["dataset_id"].value_counts())
print(f"Total authentic cohort records: {len(cohort_df)}")

image_groups_df = pd.read_parquet("artifacts/manifests/global_image_group_manifest.parquet")
print("
Image partition counts:")
print(image_groups_df["combiner_partition"].value_counts())
print(f"Held-out test images (zero leakage): {len(image_groups_df[image_groups_df['combiner_partition'] == 'held_out_test'])}")


## 4. Authentic Knowledge Corpus Verification

In [ ]:
from pathlib import Path
from src.retrieval.corpus import AuthenticKnowledgeCorpus

corpus_path = Path("artifacts/corpora/authentic_knowledge_corpus.parquet")
if not corpus_path.exists():
    print("Building authentic knowledge corpus from raw datasets...")
    !python scripts/build_authentic_corpus.py

corpus = AuthenticKnowledgeCorpus.load_parquet(corpus_path)
print(f"Total authentic documents in knowledge corpus: {len(corpus)}")
print(f"Split eligibility breakdown:")
train_docs = corpus.filter_by_eligibility("train_only")
eval_docs = corpus.filter_by_eligibility("eval_held_out")
print(f"  train_only:    {len(train_docs)}")
print(f"  eval_held_out: {len(eval_docs)}")


## 4.1 InfoSeek Image Materialization (Optional HF_TOKEN Stream)
If `HF_TOKEN` is set and you have accepted terms at https://huggingface.co/datasets/ychenNLP/oven, this downloads the required InfoSeek images to achieve 100% completion (0 skipped).

In [ ]:
# Acquire InfoSeek entity images
import os

# 1. Automatically load from Kaggle Secrets if available
if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        secret_val = user_secrets.get_secret("HF_TOKEN")
        if secret_val:
            os.environ["HF_TOKEN"] = secret_val
            os.environ["HUGGING_FACE_HUB_TOKEN"] = secret_val
    except Exception:
        pass

# 2. Acquire InfoSeek images (fast archive via BByrneLab/BERAG-demo-InfoSeek-images + ychenNLP stream if token present)
print(f"Acquiring InfoSeek entity images for Tier {TARGET_TIER}...")
!python scripts/acquire_infoseek_images.py --tier $TARGET_TIER


## 5. Sequential VLM Generation: Model 1 — LLaVA 1.5 7B

In [ ]:
# Run LLaVA-1.5-7B Generation (Tier 1000)
!python scripts/run_full_benchmark_generation.py --tier $TARGET_TIER --models llava_1.5_7b

# Explicit VRAM Cleanup
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("LLaVA-1.5-7B generation completed. VRAM cleared.")


## 6. Sequential VLM Generation: Model 2 — Qwen2-VL 7B

In [ ]:
# Run Qwen2-VL-7B Generation (Tier 1000)
!python scripts/run_full_benchmark_generation.py --tier $TARGET_TIER --models qwen2_vl_7b

# Explicit VRAM Cleanup
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Qwen2-VL-7B generation completed. VRAM cleared.")


## 7. Sequential VLM Generation: Model 3 — InternVL2 2B

In [ ]:
# Run InternVL2-2B Generation (Tier 1000)
!python scripts/run_full_benchmark_generation.py --tier $TARGET_TIER --models internvl2_2b

# Explicit VRAM Cleanup
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("InternVL2-2B generation completed. VRAM cleared.")


## 8. Multi-Axis Consistency Diagnostics & Calibrated Combiner Pipeline

In [ ]:
# Execute full diagnostic pipeline for LLaVA-1.5-7B
!python scripts/run_consistency_pipeline.py --tier $TARGET_TIER --model llava_1.5_7b

# Execute full diagnostic pipeline for Qwen2-VL-7B
!python scripts/run_consistency_pipeline.py --tier $TARGET_TIER --model qwen2_vl_7b

# Execute full diagnostic pipeline for InternVL2-2B
!python scripts/run_consistency_pipeline.py --tier $TARGET_TIER --model internvl2_2b


## 9. Component Ablation Study (S_P, S_R, S_U, Pairwise, Full XMCA-Net)

In [ ]:
# Run Component Ablation for LLaVA
!python scripts/run_component_ablation.py --features_file artifacts/features/tier_${TARGET_TIER}_llava_1.5_7b_features.parquet --output_file artifacts/reports/tier_${TARGET_TIER}_llava_ablation_report.json --tier ${TARGET_TIER} --model llava_1.5_7b

# Run Component Ablation for Qwen2-VL
!python scripts/run_component_ablation.py --features_file artifacts/features/tier_${TARGET_TIER}_qwen2_vl_7b_features.parquet --output_file artifacts/reports/tier_${TARGET_TIER}_qwen2_vl_ablation_report.json --tier ${TARGET_TIER} --model qwen2_vl_7b

# Run Component Ablation for InternVL2
!python scripts/run_component_ablation.py --features_file artifacts/features/tier_${TARGET_TIER}_internvl2_2b_features.parquet --output_file artifacts/reports/tier_${TARGET_TIER}_internvl2_ablation_report.json --tier ${TARGET_TIER} --model internvl2_2b


## 10. Research Artifact Summary & Provenance Verification

In [ ]:
import json
from pathlib import Path

reports = list(Path("artifacts/reports").glob("*.json"))
print(f"Generated {len(reports)} evaluation and ablation reports:")
for r in reports:
    print(f"  - {r.name}")
    try:
        with open(r) as f:
            data = json.load(f)
            if "detection_metrics" in data:
                print(f"    Metrics: {data['detection_metrics']}")
    except Exception as e:
        pass

print("
All research artifacts successfully generated.")


## 11. Publication Artifact & LaTeX Asset Synthesis
Compiles camera-ready LaTeX tables, vector calibration diagrams, ablation plots, and comprehensive benchmark report.

In [ ]:
# Generate publication LaTeX tables, figures, and benchmark report
!python scripts/generate_paper_artifacts.py --tier $TARGET_TIER

from pathlib import Path
report_md = Path("artifacts/reports/publication/BENCHMARK_REPORT.md")
if report_md.exists():
    print("
" + "="*60)
    print(report_md.read_text(encoding="utf-8"))
